In [1]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Воссоздаём архитектуру
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 100)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model = model.to(device)
model.eval()


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [12]:
from AircraftDataset import AircraftDataset
from torchvision import transforms
from torch.utils.data import DataLoader

transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = AircraftDataset('data/fgvc-aircraft/train.csv', 'data/fgvc-aircraft/images', transform_test)

test_dataset = AircraftDataset('data/fgvc-aircraft/test.csv', 'data/fgvc-aircraft/images', transform_test)
test_loader = DataLoader(test_dataset, batch_size=32)


In [10]:
from tqdm import tqdm

correct, total = 0, 0
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test accuracy: {correct / total:.4f}")


100%|██████████| 105/105 [01:10<00:00,  1.50it/s]

Test accuracy: 0.7147


In [21]:
from OneImage import predict_image

# model загружен и перенесён на device

class_mapping = train_dataset.class_to_idx
result = predict_image("data/fgvc-aircraft/images/0063284.jpg", model, train_dataset.class_to_idx, device)
print("Predicted class:", result)


Predicted class: 737-400
